In [3]:
#!pip install datasets pypdf


In [7]:
# Import necessary libraries
from transformers import AutoTokenizer, AutoModel
from pypdf import PdfReader
import torch
import re

# Load a pre-trained model for embedding generation (e.g., SentenceTransformers)
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
model = AutoModel.from_pretrained(embedding_model_name)

# Function to extract text from a PDF
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

# Function to clean and preprocess text
def clean_text(text):
    text = re.sub(r"\s+", " ", text)  # Normalize whitespace
    return text.strip()

# Function to split text into chunks for embedding
def split_text_into_chunks(text, max_tokens=200):
    sentences = re.split(r'(?<=[.!?]) +', text)  # Split by sentence boundaries
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk.split()) + len(sentence.split()) <= max_tokens:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

# Function to generate embeddings for text chunks
def generate_embeddings(chunks):
    embeddings = []
    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, padding=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
            # Pool embeddings (e.g., mean pooling)
            embeddings.append(outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy())
    return embeddings

# Main pipeline to extract embeddings from a PDF
def extract_embeddings_from_pdf(pdf_path):
    # Step 1: Extract and clean text
    raw_text = extract_text_from_pdf(pdf_path)
    cleaned_text = clean_text(raw_text)

    # Step 2: Split text into chunks
    chunks = split_text_into_chunks(cleaned_text)

    # Step 3: Generate embeddings
    embeddings = generate_embeddings(chunks)

    return chunks, embeddings

# Example Usage
pdf_path = "example.pdf"  # Replace with your PDF file path

print("\n--- Extracting embeddings from PDF ---")
chunks, embeddings = extract_embeddings_from_pdf(pdf_path)

# Print the results
for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
    print(f"Chunk {i+1}: {chunk}")
    print(f"Embedding {i+1}: {embedding[:5]}... (truncated)")
    print()



--- Extracting embeddings from PDF ---
Chunk 1: natural-resources.canada.ca /our-natural-resources/energy-sources-distribution/electricity-infrastru… Powering Canada’s Future: A Clean Electricity Strategy 155-197 minutes Table of Contents Foreword – Clean Electricity Strategy 1.0 The Case for Clean Electricity 1.1 Laying Out a Clean Electricity Strategy for Canada 1.2 A Strategy informed by extensive engagement, electricity sector experts, and Indigenous energy leaders 1.3 Key Guiding Principles 2.0 Toward the Grid of the Future 2.1 Global Context 2.2 Canadian Context 2.3 Regional Context 3.0 Federal Action 3.1 Focus Area 1: Growing the Grid and Managing Demand 3.2 Focus Area 2: Providing Policy Certainty and Smoothing the Path 3.3 Focus Area 3: Collaborating on Tailored Approaches for Every Region 4.0 Next Steps Annex 1 – Canada Electricity Advisory Council Recommendations Annex 2 – Wah-ila-toos Indigenous Council Recommendations List of Figures, Tables and Boxes Foreword – Clean Ele